In [ ]:
!pip install -q insightface onnxruntime opencv-python scikit-learn matplotlib numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 4.8 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 42.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 73.1 MB/s eta 0:00:00:00:0100:01
  Created wheel for insightface: filename=insightface-0.7.3-cp312-cp312-linux_x86_64.whl size=1071490 sha256=8c698ec40948d024b2e6dfffc363e947bbe3086c46750687e6a0518f506bf730
  Stored in directory: /root/.cache/pip/wheels/73/3c/e2/6d4815e8a8b33a2006554d65ce0d1f973e768f4c7a222fa675
Successfully built insightface


In [2]:
import os
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from insightface.app import FaceAnalysis
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# ---- YOUR ACTUAL DATA PATHS
FOLDERS = {
    "uncanny": Path(r"C:\Users\maryc\Downloads\881project\uncannyrobot"),
    "stylized": Path(r"C:\Users\maryc\Downloads\881project\stylishedrobot"),
    "human": Path(r"C:\Users\maryc\Downloads\881project\humans_ffhq"),
}

# ---- Load face model
app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=-1)  # CPU mode

def best_face_embedding(img_bgr):
    faces = app.get(img_bgr)
    if not faces:
        return None
    faces = sorted(
        faces,
        key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]),
        reverse=True
    )
    return faces[0].embedding

rows = []
embs = []

# ---- Extract embeddings
for group, folder in FOLDERS.items():
    for p in sorted(folder.glob("*")):
        if p.suffix.lower() not in [".jpg", ".jpeg", ".png", ".webp"]:
            continue
        img = cv2.imread(str(p))
        if img is None:
            rows.append({"path": str(p), "group": group, "status": "read_fail"})
            continue
        emb = best_face_embedding(img)
        if emb is None:
            rows.append({"path": str(p), "group": group, "status": "no_face"})
            continue
        embs.append(emb)
        rows.append({"path": str(p), "group": group, "status": "ok"})

df = pd.DataFrame(rows)

ok_rows = df[df["status"] == "ok"].copy().reset_index(drop=True)
X = np.vstack(embs)

print(df["status"].value_counts())

# ---- Split embeddings by group
human_X = X[ok_rows["group"] == "human"]
uncanny_X = X[ok_rows["group"] == "uncanny"]
stylized_X = X[ok_rows["group"] == "stylized"]

# ---- Human centroid
human_centroid = human_X.mean(axis=0, keepdims=True)

def dist_to_centroid(A):
    return euclidean_distances(A, human_centroid).reshape(-1)

print("\nMean distance to human centroid:")
print("uncanny:", dist_to_centroid(uncanny_X).mean())
print("stylized:", dist_to_centroid(stylized_X).mean())

# ---- PCA visualization
pca = PCA(n_components=2)
X2 = pca.fit_transform(X)

plt.figure()
for g in ["human", "uncanny", "stylized"]:
    idx = ok_rows["group"] == g
    plt.scatter(X2[idx, 0], X2[idx, 1], label=g, s=20)

plt.legend()
plt.title("Embedding space (PCA)")
plt.savefig("methodA_pca.png", dpi=200)
plt.show()


download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:03<00:00, 79028.44KB/s]
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)


KeyError: 'status'